In [35]:
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise.accuracy import rmse, mae
import numpy as np
import math
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [2]:


# Đọc file ratings.dat
ratings_df = pd.read_csv('ratings.dat', sep='::', engine='python', 
                      names=['UserId', 'MovieId', 'Rating', 'Timestamp'])

# Đọc file tags.dat
tags_df = pd.read_csv('tags.dat', sep='::', engine='python', 
                   names=['UserId', 'MovieId', 'Tag', 'Timestamp'])

# Đọc file movies.dat
movies_df = pd.read_csv('movies.dat', sep='::', engine='python', 
                     names=['MovieId', 'Title', 'Genres'])

In [3]:
ratings_df.describe()

,UserId,MovieId,Rating,Timestamp
count,1.000005e+07,1.000005e+07,1.000005e+07,1.000005e+07
mean,3.586986e+04,4.120291e+03,3.512422e+00,1.032606e+09
std,2.058534e+04,8.938402e+03,1.060418e+00,1.159640e+08
min,1.000000e+00,1.000000e+00,5.000000e-01,7.896520e+08
25%,1.812300e+04,6.480000e+02,3.000000e+00,9.467659e+08
50%,3.574050e+04,1.834000e+03,4.000000e+00,1.035476e+09
75%,5.360800e+04,3.624000e+03,4.000000e+00,1.126749e+09
max,7.156700e+04,6.513300e+04,5.000000e+00,1.231132e+09


In [4]:
ratings_df.duplicated(subset=['UserId', 'MovieId']).sum()

0

In [5]:
# Lọc user >= 5 ratings
user_counts = ratings_df.groupby('UserId').size()
ratings_df = ratings_df[ratings_df['UserId'].isin(user_counts[user_counts >= 5].index)]

# Lọc movie >= 5 ratings
movie_counts = ratings_df.groupby('MovieId').size()
ratings_df = ratings_df[ratings_df['MovieId'].isin(movie_counts[movie_counts >= 5].index)]

In [6]:
n_users = ratings_df['UserId'].nunique()
n_movies = ratings_df['MovieId'].nunique()
n_ratings = len(ratings_df)

sparsity = 1 - (n_ratings / (n_users * n_movies))
print("Sparsity:", sparsity)

Sparsity: 0.9859661030476576


In [7]:
ratings_df.groupby('UserId').size().describe()


count    69878.000000
mean       143.089613
std        216.543129
min         18.000000
25%         35.000000
50%         69.000000
75%        156.000000
max       7240.000000
dtype: float64

In [8]:
ratings_df.groupby('MovieId').size().describe()

count    10196.000000
mean       980.660651
std       2536.847198
min          5.000000
25%         41.000000
50%        152.000000
75%        684.250000
max      34864.000000
dtype: float64

In [9]:
movies_df.head()
tags_df.head()
ratings_df.head()

,UserId,MovieId,Rating,Timestamp
0,1,122,5.0,838985046
1,1,185,5.0,838983525
2,1,231,5.0,838983392
3,1,292,5.0,838983421
4,1,316,5.0,838983392


In [10]:
tags_df.isnull().sum()

UserId        0
MovieId       0
Tag          16
Timestamp     0
dtype: int64

In [11]:
# Loại bỏ các hàng có dữ liệu thiếu
ratings_df.dropna(inplace=True)
tags_df.dropna(inplace=True)
movies_df.dropna(inplace=True)

In [12]:
ratings_df['Timestamp'] = pd.to_datetime(ratings_df['Timestamp'], unit='s')
tags_df['Timestamp'] = pd.to_datetime(tags_df['Timestamp'], unit='s')

In [13]:
ratings_df.head()

,UserId,MovieId,Rating,Timestamp
0,1,122,5.0,1996-08-02 11:24:06
1,1,185,5.0,1996-08-02 10:58:45
2,1,231,5.0,1996-08-02 10:56:32
3,1,292,5.0,1996-08-02 10:57:01
4,1,316,5.0,1996-08-02 10:56:32


In [14]:


svd_df = ratings_df[["UserId", "MovieId", "Rating", "Timestamp"]].copy()

# Chỉ xem rating >= 4 là positive để đánh giá ranking
positive_df = svd_df[svd_df["Rating"] >= 4.0].copy()

# Chỉ giữ user có ít nhất 2 positive items
pos_counts = positive_df.groupby("UserId").size()
eval_users = pos_counts[pos_counts >= 2].index

positive_df = positive_df[positive_df["UserId"].isin(eval_users)].copy()
svd_df = svd_df[svd_df["UserId"].isin(eval_users)].copy()

# Hold out 1 positive item cuối cùng của mỗi user
heldout_df = (
    positive_df
    .sort_values(["UserId", "Timestamp"])
    .groupby("UserId")
    .tail(1)[["UserId", "MovieId", "Rating"]]
    .copy()
)

heldout_pairs = set(zip(heldout_df["UserId"], heldout_df["MovieId"]))

# Train data = bỏ các pair holdout ra
train_df = svd_df[
    ~svd_df[["UserId", "MovieId"]].apply(tuple, axis=1).isin(heldout_pairs)
].copy()

reader = Reader(rating_scale=(0.5, 5.0))
train_data = Dataset.load_from_df(train_df[["UserId", "MovieId", "Rating"]], reader)
trainset = train_data.build_full_trainset()

svd_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42 
)

svd_model.fit(trainset)

print("Train interactions:", len(train_df))
print("Held-out users:", heldout_df["UserId"].nunique())
print("Held-out items:", len(heldout_df))

Train interactions: 9923162
Held-out users: 69715
Held-out items: 69715


In [15]:
def recommend_for_user_svd(user_id, user_history, movies_df, model, top_n=10):
    watched = set(user_history)  # lấy từ DB thật
    all_movies = movies_df["MovieId"].unique()

    candidates = [mid for mid in all_movies if mid not in watched]

    preds = []
    for mid in candidates:
        est = model.predict(uid=user_id, iid=mid).est
        preds.append((mid, est))

    recs = (
        pd.DataFrame(preds, columns=["MovieId", "pred_score"])
        .merge(movies_df, on="MovieId", how="left")
        .sort_values("pred_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    return recs

In [16]:
def recommend_for_user_svd(user_id, train_df, movies_df, model, top_n=10):
    watched = set(train_df.loc[train_df["UserId"] == user_id, "MovieId"])
    all_movies = movies_df["MovieId"].unique()

    candidates = [mid for mid in all_movies if mid not in watched]

    preds = []
    for mid in candidates:
        est = model.predict(uid=user_id, iid=mid).est
        preds.append((mid, est))

    recs = (
        pd.DataFrame(preds, columns=["MovieId", "pred_score"])
        .merge(movies_df, on="MovieId", how="left")
        .sort_values("pred_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    return recs

In [17]:
recommend_for_user_svd(
    user_id=2,
    train_df=train_df,
    movies_df=movies_df,
    model=svd_model,
    top_n=10
)

,MovieId,pred_score,Title,Genres
0,4993,4.415926,"Lord of the Rings: The Fellowship of the Ring,...",Action|Adventure|Fantasy
1,7153,4.399007,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Fantasy
2,5952,4.343117,"Lord of the Rings: The Two Towers, The (2002)",Action|Adventure|Fantasy
3,6896,4.189323,Shoah (1985),Documentary|War
4,4973,4.154091,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance
5,1148,4.112592,Wallace & Gromit: The Wrong Trousers (1993),Animation|Children|Comedy|Crime
6,44555,4.106231,"Lives of Others, The (Das Leben der Anderen) (...",Drama
7,2019,4.090304,Seven Samurai (Shichinin no samurai) (1954),Action|Drama
8,5618,4.086998,Spirited Away (Sen to Chihiro no kamikakushi) ...,Adventure|Animation|Children|Fantasy
9,745,4.076661,Wallace & Gromit: A Close Shave (1995),Animation|Children|Comedy


In [18]:


def evaluate_ranking_svd_sampled(model, train_df, heldout_df, movies_df, k=10, n_neg=100, random_state=42):
    rng = np.random.default_rng(random_state)
    all_movies = set(movies_df["MovieId"].unique())

    train_user_items = (
        train_df.groupby("UserId")["MovieId"]
        .apply(set)
        .to_dict()
    )

    hr_scores = []
    ndcg_scores = []
    recommended_items = set()

    for row in heldout_df.itertuples(index=False):
        user_id = row.UserId
        true_item = row.MovieId

        watched = train_user_items.get(user_id, set())

        # Lấy negative samples thay vì quét toàn bộ item
        negatives = list(all_movies - watched - {true_item})

        if len(negatives) > n_neg:
            negatives = rng.choice(negatives, size=n_neg, replace=False).tolist()

        candidates = negatives + [true_item]

        preds = []
        for item_id in candidates:
            est = model.predict(uid=user_id, iid=item_id).est
            preds.append((item_id, est))

        preds.sort(key=lambda x: x[1], reverse=True)
        top_k = preds[:k]
        top_k_items = [item for item, _ in top_k]

        recommended_items.update(top_k_items)

        if true_item in top_k_items:
            hr_scores.append(1)
            rank = top_k_items.index(true_item) + 1
            ndcg_scores.append(1 / math.log2(rank + 1))
        else:
            hr_scores.append(0)
            ndcg_scores.append(0)

    hr_k = np.mean(hr_scores)
    ndcg_k = np.mean(ndcg_scores)
    coverage_k = len(recommended_items) / movies_df["MovieId"].nunique()

    return {
        f"HR@{k}": hr_k,
        f"NDCG@{k}": ndcg_k,
        f"Coverage@{k}": coverage_k
    }

In [19]:
metrics_10 = evaluate_ranking_svd_sampled(
    model=svd_model,
    train_df=train_df,
    heldout_df=heldout_df,
    movies_df=movies_df,
    k=10,
    n_neg=100
)

metrics_10

{'HR@10': 0.38402065552607045,
 'NDCG@10': 0.22900656756807478,
 'Coverage@10': 0.6010673157944013}

#   COLD-START

In [20]:
movies_cb = movies_df[["MovieId", "Title", "Genres"]].copy()
movies_cb["Genres"] = movies_cb["Genres"].fillna("")
movies_cb.head()

,MovieId,Title,Genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [22]:
genre_features = movies_cb["Genres"].str.get_dummies(sep="|")
genre_features.head(10)

,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0
6,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
7,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0


In [23]:
movie_content_df = pd.concat(
    [movies_cb[["MovieId", "Title", "Genres"]], genre_features],
    axis=1
)

movie_content_df.head()

,MovieId,Title,Genres,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,0,0,1,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children|Fantasy,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,0,0,0,0,0,1,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


PHẦN NÀY ĐỂ LẠI ĐỂ Content-based similarity gợi ý genre_matrix

In [24]:
genre_cols = genre_features.columns.tolist()
genre_matrix = movie_content_df[genre_cols].values

print("Số phim:", movie_content_df.shape[0])
print("Số genre features:", len(genre_cols))
print("Shape genre_matrix:", genre_matrix.shape)

Số phim: 10681
Số genre features: 20
Shape genre_matrix: (10681, 20)


In [25]:
movie_content_df[movie_content_df["Title"].str.contains("Matrix", case=False, na=False)].head(3)

,MovieId,Title,Genres,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
2487,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,0
6258,6365,"Matrix Reloaded, The (2003)",Action|Sci-Fi|Thriller,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,0
6820,6934,"Matrix Revolutions, The (2003)",Action|Sci-Fi|Thriller,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,0


In [28]:
popular_movies = (
    ratings_df.groupby("MovieId")
    .agg(
        avg_rating=("Rating", "mean"),
        rating_count=("Rating", "count")
    )
    .reset_index()
)

# lọc item quá ít rating để tránh nhiễu
popular_movies = popular_movies[popular_movies["rating_count"] >= 20].copy()

# popularity score đơn giản
popular_movies["pop_score"] = (
    popular_movies["avg_rating"] * np.log1p(popular_movies["rating_count"])
)

popular_movies = (
    popular_movies.merge(movies_df[["MovieId", "Title", "Genres"]], on="MovieId", how="left")
    .sort_values("pop_score", ascending=False)
    .reset_index(drop=True)
)

popular_movies.head()

,MovieId,avg_rating,rating_count,pop_score,Title,Genres
0,318,4.457238,31126,46.113834,"Shawshank Redemption, The (1994)",Drama
1,527,4.363483,25777,44.321104,Schindler's List (1993),Drama|War
2,50,4.367142,24037,44.053073,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
3,593,4.204200,33668,43.825978,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
4,858,4.415085,19814,43.683713,"Godfather, The (1972)",Crime|Drama


In [29]:
def recommend_new_user_by_genres_scored(preferred_genres, popular_movies, top_n=10, alpha=0.7):
    df = popular_movies.copy()
    preferred_set = set(g.lower() for g in preferred_genres)

    def count_matches(genres_str):
        genres = set(genres_str.lower().split("|")) if pd.notna(genres_str) and genres_str else set()
        return len(preferred_set.intersection(genres))

    df["genre_match_count"] = df["Genres"].apply(count_matches)
    df = df[df["genre_match_count"] > 0].copy()

    # normalize genre match
    gmin, gmax = df["genre_match_count"].min(), df["genre_match_count"].max()
    pmin, pmax = df["pop_score"].min(), df["pop_score"].max()

    if gmax > gmin:
        df["genre_match_norm"] = (df["genre_match_count"] - gmin) / (gmax - gmin)
    else:
        df["genre_match_norm"] = 1.0

    if pmax > pmin:
        df["pop_norm"] = (df["pop_score"] - pmin) / (pmax - pmin)
    else:
        df["pop_norm"] = 1.0

    df["final_score"] = alpha * df["genre_match_norm"] + (1 - alpha) * df["pop_norm"]

    recs = (
        df.sort_values(["final_score", "rating_count"], ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    return recs[[
        "MovieId", "Title", "Genres",
        "genre_match_count", "avg_rating", "rating_count", "pop_score", "final_score"
    ]]

In [32]:
recommend_new_user_by_genres_scored(
    preferred_genres=["Action", "Sci-Fi", "Thriller","Drama","Mystery"],
    popular_movies=popular_movies,
    top_n=10,
    alpha=0.7
)

,MovieId,Title,Genres,genre_match_count,avg_rating,rating_count,pop_score,final_score
0,198,Strange Days (1995),Action|Crime|Drama|Mystery|Sci-Fi|Thriller,5,3.413455,5604,29.462942,0.880456
1,4878,Donnie Darko (2001),Drama|Mystery|Sci-Fi|Thriller,4,4.045675,7225,35.947604,0.752012
2,5445,Minority Report (2002),Action|Crime|Mystery|Sci-Fi|Thriller,4,3.659857,11038,34.070302,0.738534
3,292,Outbreak (1995),Action|Drama|Sci-Fi|Thriller,4,3.418414,16075,33.107619,0.731623
4,8665,"Bourne Supremacy, The (2004)",Action|Adventure|Drama|Mystery|Thriller,4,3.779786,5229,32.363156,0.726278
5,48774,Children of Men (2006),Action|Adventure|Drama|Sci-Fi|Thriller,4,3.906606,2634,30.770922,0.714847
6,2058,"Negotiator, The (1998)",Action|Crime|Drama|Mystery|Thriller,4,3.591252,4767,30.416766,0.712304
7,27773,Oldboy (2005),Action|Adventure|Crime|Drama|Mystery|Thriller,4,4.073142,1709,30.321483,0.711620
8,1264,Diva (1981),Action|Drama|Mystery|Romance|Thriller,4,4.045315,1633,29.930424,0.708812
9,1909,"X-Files: Fight the Future, The (1998)",Action|Crime|Mystery|Sci-Fi|Thriller,4,3.363168,6497,29.526088,0.705909


In [34]:
popular_movies.to_csv("data/popular_movies.csv", index=False)

In [36]:
svd_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42 
)

svd_model.fit(trainset)

with open("data/svd_model.pkl", "wb") as f:
    pickle.dump(svd_model, f)

print("Đã lưu model vào data/svd_model.pkl")

Đã lưu model vào data/svd_model.pkl
